Author: Danny Lillard

Date: 2026-04-03

Desc: Testing return of algorithm we will look at the following:

- Buy and hold
- Naive algo
- Clustering algo

In [35]:
# buy and hold
import pandas as pd

raw_data_df = pd.read_csv("quarterly_clusters_model/aa_daily_data.csv")
raw_data_df['timestamp'] = pd.to_datetime(raw_data_df['timestamp'])

start_date = pd.to_datetime("2022-01-03")
end_date = pd.to_datetime("2022-03-31")

start_and_stop_df = raw_data_df[raw_data_df['timestamp'].isin([start_date, end_date])]

# Get the unique tickers in the dataset
tickers = start_and_stop_df['ticker'].unique()

# Calculate the return for each ticker
returns = []
for ticker in tickers:
    ticker_data = start_and_stop_df[start_and_stop_df['ticker'] == ticker]
    ticker_data = ticker_data.sort_values('timestamp')
    
    # Get the opening price on the start date and the closing price on the end date
    start_price = ticker_data[ticker_data['timestamp'] == start_date]['open'].values[0]
    end_price = ticker_data[ticker_data['timestamp'] == pd.to_datetime("2022-03-31")]['close'].values[0]
    
    # Calculate the return
    return_pct = (end_price - start_price) / start_price
    returns.append((ticker,return_pct))

# save returns to csv
returns_df = pd.DataFrame(returns, columns=['ticker', 'return'])
returns_df.to_csv("buy_and_hold_returns.csv", index=False)

# calculate overall return
overall_return = sum([r[1] for r in returns]) / len(returns)
print(f"Overall Buy and Hold Return: {overall_return:.2%}")

Overall Buy and Hold Return: -6.41%


In [ ]:
# seeing returns for naive algo
naive_returns_df = pd.read_csv("signals_and_returns.csv")

naive_returns_df.columns = ['timestamp', 'ticker', 'signal', 'return']

# calculate overall return for naive algo
overall_naive_return = naive_returns_df['return'].mean()

KeyError: 'return'

In [43]:
# seeing returns for naive algo
naive_returns_df = pd.read_csv("signals_and_returns.csv",header=None)

naive_returns_df.columns = ['timestamp', 'ticker', 'signal', 'return']

naive_returns_df['timestamp'] = pd.to_datetime(naive_returns_df['timestamp'])

naive_returns_df = naive_returns_df.drop('return',axis=1)

# need to zip with raw data to get open prices
naive_returns_and_raw_df = naive_returns_df.merge(raw_data_df[['timestamp', 'ticker', 'open']], on=['timestamp', 'ticker'], how='left')

# first day buy signals
held_stocks = naive_returns_and_raw_df[(naive_returns_and_raw_df['signal'] == 'BUY') & (naive_returns_and_raw_df['timestamp'] == '2022-01-03')][['ticker','open']]



In [45]:
held_stocks

,ticker,open
62,TEN,7.2500
248,URBN,29.7600
496,MG,7.4100
930,NBHC,44.3500
992,CYCN,1.7200
...,...,...
11904,ULBI,6.0900
12152,BTU,10.4100
12214,NRC,42.0545
12462,GE,95.2200


In [ ]:
# for each day, calculate returns for held stocks
for date in pd.date_range(start_date, end_date):
    daily_data = naive_returns_and_raw_df[raw_data_df['timestamp'] == date]

    # three cases, buy sell and hold
    buys = daily_data[daily_data['signal'] == 'BUY']
    sells = daily_data[daily_data['signal'] == 'SELL']
    holds = daily_data[daily_data['signal'] == 'HOLD']

    
    
    held_stocks = daily_data[['ticker', 'open_x']].rename(columns={'open_x': 'open'})